In [3]:
import sys
from pathlib import Path

ROOT = Path.cwd()

while not (ROOT / "config.py").exists():

    if ROOT.parent == ROOT:
        raise RuntimeError("Project root not found")

    ROOT = ROOT.parent

sys.path.append(str(ROOT))

print("PROJECT ROOT:", ROOT)

PROJECT ROOT: c:\Users\nagal\Documents\AI\rag-benchmark


In [ ]:
CHROMA_PATH = Path(config.CHROMA_PERSIST_DIR)

print("Deleting:", CHROMA_PATH)

if CHROMA_PATH.exists():
    shutil.rmtree(CHROMA_PATH)
    print("✅ Chroma deleted")
else:
    print("ℹ️ Chroma does not exist")

In [ ]:
BM25_INDEX = Path("vectorless_rag/bm25_index.pkl")
BM25_MANIFEST = Path("vectorless_rag/bm25_manifest.json")

for path in [BM25_INDEX, BM25_MANIFEST]:

    if path.exists():
        path.unlink()
        print(f"Deleted {path}")
        

In [ ]:
print("Running preprocessing...")

data = run_preprocessing_pipeline()

print()

print("Parents :", len(data["parents"]))
print("Children:", len(data["children"]))

In [ ]:
parent_ids = {
    p["chunk_id"]
    for p in data["parents"]
}

broken = []

for child in data["children"]:

    if child["parent_id"] not in parent_ids:

        broken.append(
            child["chunk_id"]
        )

print(
    "Broken references:",
    len(broken)
)

In [ ]:
print("Building vector index...")

collection = index_chunks(data)

print()

print(
    "Vectors in Chroma:",
    collection.count()
)

In [ ]:
print("Building BM25...")

build_bm25_index(data)

print("✅ BM25 rebuilt")

from pathlib import Path

print(Path.cwd())

In [ ]:
from pathlib import Path

BM25_INDEX = (
    ROOT /
    "vectorless_rag" /
    "bm25_index.pkl"
)

BM25_MANIFEST = (
    ROOT /
    "vectorless_rag" /
    "bm25_manifest.json"
)

for file in [BM25_INDEX, BM25_MANIFEST]:

    if file.exists():
        file.unlink()
        print(f"✅ Deleted: {file}")
    else:
        print(f"ℹ️ Not found: {file}")

In [ ]:
print("BM25 Index Exists   :", BM25_INDEX.exists())
print("BM25 Manifest Exists:", BM25_MANIFEST.exists())

In [ ]:
from data_loader import run_preprocessing_pipeline
from vectorless_rag.indexer import build_bm25_index

print("Loading processed data...")

data = run_preprocessing_pipeline()

print(
    f"Parents: {len(data['parents'])}"
)
print(
    f"Children: {len(data['children'])}"
)

print("\nBuilding BM25...")

build_bm25_index(data)

print("✅ BM25 build complete")

In [ ]:
from vector_rag.indexer import (
    index_chunks,
    get_chroma_collection
)

collection = get_chroma_collection()

print(
    "Chroma count:",
    collection.count()
)

In [ ]:
results = collection.query(
    query_texts=["microsoft revenue"],
    n_results=3,
    include=[
        "documents",
        "metadatas"
    ]
)

for meta in results["metadatas"][0]:

    print(meta)
    print()

In [4]:
from vector_rag.pipeline import (
    VectorRAGPipeline
)

from vector_rag.retriever import (
    retrieve
)

rag = VectorRAGPipeline()

Loading Vector_pipeline...
AAAYYYOOOOOO
Loading ReRanker...
ReRanker loaded in 29.92s
Vector_pipeline loaded in 34.57s
🔧 Initialising Vector RAG Pipeline...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

✅ ChromaDB loaded — 20669 child vectors
Mistral client ready - model: mistral-medium-latest
✅ Vector RAG ready — 5841 parents in lookup



In [5]:
result = retrieve(
    query="What was NVIDIAs revenue in 2025?",
    collection=rag.collection,
    parent_lookup=rag.parent_lookup,
    top_k=5
)

for i, chunk in enumerate(result["chunks"]):

    print("=" * 80)

    print("Rank:", i + 1)

    print(
        "Metadata Company:",
        chunk["metadata"]["company"]
    )

    parent_id = (
        chunk["metadata"]["parent_id"]
    )

    print(
        "Parent Company:",
        rag.parent_lookup[parent_id]["company"]
    )

    print(
        "Parent ID:",
        parent_id
    )


===== COMPANY MATCH DEBUG =====
QUESTION: what was nvidias revenue in 2025?
MATCHED COMPANY: NVIDIA
RAW: What was NVIDIAs revenue in 2025?
PROCESSED: What was NVIDIAs revenue in 2025?

ORIGINAL : What was NVIDIAs revenue in 2025?
COMPANY  : NVIDIA
YEAR     : 2025
USED FOR SEARCH:
What was NVIDIAs revenue in 2025?

SEARCH QUERY: Represent this sentence for searching relevant passages: What was NVIDIAs revenue in 2025?
🔁 Loading reranker: BAAI/bge-reranker-large


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

✅ Reranker ready

========== DEBUG ==========
PARENT ID: parent_4129
CHILD COMPANY: NVIDIA
LOOKUP COMPANY: NVIDIA
LOOKUP PAGE: 69
LOOKUP CHUNK: parent_4129

========== DEBUG ==========
PARENT ID: parent_4043
CHILD COMPANY: NVIDIA
LOOKUP COMPANY: NVIDIA
LOOKUP PAGE: 53
LOOKUP CHUNK: parent_4043

========== DEBUG ==========
PARENT ID: parent_4172
CHILD COMPANY: NVIDIA
LOOKUP COMPANY: NVIDIA
LOOKUP PAGE: 78
LOOKUP CHUNK: parent_4172

========== DEBUG ==========
PARENT ID: parent_4170
CHILD COMPANY: NVIDIA
LOOKUP COMPANY: NVIDIA
LOOKUP PAGE: 77
LOOKUP CHUNK: parent_4170

========== DEBUG ==========
PARENT ID: parent_4041
CHILD COMPANY: NVIDIA
LOOKUP COMPANY: NVIDIA
LOOKUP PAGE: 52
LOOKUP CHUNK: parent_4041
Rank: 1
Metadata Company: NVIDIA
Parent Company: NVIDIA
Parent ID: parent_4129
Rank: 2
Metadata Company: NVIDIA
Parent Company: NVIDIA
Parent ID: parent_4043
Rank: 3
Metadata Company: NVIDIA
Parent Company: NVIDIA
Parent ID: parent_4172
Rank: 4
Metadata Company: NVIDIA
Parent Company: NV

In [ ]:
from pathlib import Path
import json

ROOT = Path.cwd().parent

questions_path = (
    ROOT
    / "evaluation"
    / "test_questions_test.json"
)

print("Exists:", questions_path.exists())
print("Path:", questions_path)

with open(
    questions_path,
    "r",
    encoding="utf-8"
) as f:
    questions = json.load(f)

print(
    "Questions:",
    len(questions["questions"])


from tqdm import tqdm

questions_path = (
    Path("evaluation")
    / "test_questions.json"
)

with open(
    questions_path,
    encoding="utf-8"
) as f:

    questions = json.load(f)["questions"]

dataset = []

for item in tqdm(questions):

    result = retrieve(
        query=item["question"],
        collection=rag.collection,
        parent_lookup=rag.parent_lookup,
        top_k=20
    )

    chunks = result["chunks"]

    if not chunks:
        continue

    dataset.append({

        "question":
            item["question"],

        "positive_chunk":
            chunks[0]["child_text"],

        "company":
            item["company"],

        "category":
            item["category"]
    })

print(
    "Pairs:",
    len(dataset)
)
)

SyntaxError: invalid syntax (2235672592.py, line 26)